<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.2-transient-heat/Ex08.2_02_hard_ic_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.2 · Notebook 02 — Hard-Enforced Initial Condition

**Paired with L8.2 · Dynamic Heat**

$$T_{\mathrm{trial}} = (1-t)\,f_{\mathrm{IC}}(x,y)
+ t\,x(1-x)y(1-y)\,\mathcal{N}_{\hat w}(x,y,t)$$

Both conditions are now structural, so $\mathcal{L}=\mathcal{L}_{\mathrm{PDE}}$.
At $t = 0$ the second term vanishes and the trial solution *is* the initial
field. On the four edges $x(1-x)y(1-y)$ vanishes and so does
$f_{\mathrm{IC}}$, so the boundary condition holds too — for every $t$.

One loss term, no weights, nothing to trade.

## What you will do

1. Build the trial solution and check both conditions **before** training.
2. Train on the residual alone.
3. Compare against notebook 01 instant by instant.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.2-transient-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The points

No boundary points and no initial points at all — there is nothing left for
them to enforce.

In [ ]:
C, T_END = 1.0, 1.0
N_F = 3000

set_seed(88)
model_h = MLP(n_in=3, n_hidden=40, n_layers=4)
describe(model_h, N_F)

xyt_f = to_tensor(pb.plate_spacetime_points(N_F, t_end=T_END),
                  requires_grad=True)

## 2 · The trial solution

### Your turn

In [ ]:
# TODO 1 --- the trial solution ---------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  x * (1 - x) * y * (1 - y)                    zero on all four edges
#   line 2  ->  (1 - t) * f_ic + t * f_dbc * model(xyt)      (1 - t) carries the initial field, t switches the network on
def trial(model, xyt):
    x, y, t = xyt[:, 0:1], xyt[:, 1:2], xyt[:, 2:3]
    f_ic  = torch.sin(np.pi * x) * torch.sin(np.pi * y)    # the initial field
    f_dbc = ...                                   # <- x * (1 - x) * y * (1 - y)
    return ...                                    # <- (1 - t) * f_ic + t * f_dbc * model(xyt)
# ------------------------------------------------------------------------------

In [ ]:
# Both conditions, on an UNTRAINED network. Nothing has been optimised yet.
set_seed(0)
probe = MLP(n_in=3, n_hidden=40, n_layers=4)

xy0 = initial_points(400, pb.PLATE_DOMAIN, t0=0.0, seed=3)
xyb = boundary_points_in_time(25, 10, pb.PLATE_DOMAIN, (0.0, T_END), seed=3)
with torch.no_grad():
    ic_err = np.abs(to_numpy(trial(probe, to_tensor(xy0))).ravel()
                    - pb.exact_transient(xy0[:, 0], xy0[:, 1], 0.0)).max()
    edge_err = np.abs(to_numpy(trial(probe, to_tensor(xyb)))).max()

print(f"initial condition, untrained : {ic_err:.3e}")
print(f"edge condition,    untrained : {edge_err:.3e}")
check("initial field is exact before training", ic_err, 0.0, tol=1e-12)
check("edges are exact before training", edge_err, 0.0, tol=1e-12)

**What you should see.** Two `PASS` lines. A network that knows nothing already
reproduces the initial field and the edges to machine precision — compare with
notebook 01, which needed four thousand Adam steps to approach the initial
field and never reached it.

---

## 3 · The single-term loss

### Your turn

In [ ]:
# TODO 2 --- residual and loss, one term ---------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  trial(model, xyt)                          differentiate the TRIAL solution, not the raw network
#   line 2  ->  mse(residual_hard(model_h, xyt_f))
def residual_hard(model, xyt):
    T = ...                                       # <- trial(model, xyt)
    T_t = grad(T, xyt)[:, 2:3]
    return T_t - C * (d2(T, xyt, 0) + d2(T, xyt, 1))

def loss_fn():
    return ...                                    # <- mse(residual_hard(model_h, xyt_f))
# ------------------------------------------------------------------------------

## 4 · Train

In [ ]:
history_hard = train_two_stage(model_h, loss_fn, adam_steps=4000,
                               lbfgs_steps=200, lr=1e-3)
plot_curves(history_hard, title="the plate — hard IC and BC, one loss term")
plt.show()

## 5 · Instant by instant, against notebook 01

In [ ]:
ts, rel, ab = pb.error_vs_time(model_h, c=C, trial=trial)
soft = np.load(os.path.join("Ex08.2_outputs", "nb01_soft.npz"))

print(error_table(
    [[f"{t:.1f}", f"{s:.3e}", f"{h:.3e}", f"{sa:.3e}", f"{ha:.3e}"]
     for t, s, h, sa, ha in zip(ts, soft["rel"], rel, soft["abs_err"], ab)],
    ["t", "soft rel", "hard rel", "soft abs", "hard abs"]))

fig, ax = plt.subplots(figsize=(7.4, 4.4))
ax.semilogy(ts, soft["abs_err"], "o-", lw=1.9, ms=6, color="#d94f2b",
            label="soft IC")
ax.semilogy(ts, ab, "s-", lw=1.9, ms=6, color="#1f77b4", label="hard IC")
ax.set_xlabel("t"); ax.set_ylabel("max absolute error")
ax.set_title("Where the two methods differ")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

X, Y, U = pb.slice_at_time(model_h, 0.0, trial=trial)
print(f"max |error| at t=0: {np.abs(U - pb.exact_transient(X, Y, 0.0)).max():.3e}")

**Check.** The $t = 0$ error must be at machine precision. It is not a
well-trained model, it is an identity: at $t = 0$ the trial solution has no
network in it at all.

Look at where the advantage goes as $t$ grows. **Hard enforcement helps most
where the information was**, and by late times the network has been integrating
its own residual for several time constants and the free gift at $t=0$ is a
long way behind it.

And it is not free. The factor $t\,f_{\mathrm{dbc}}$ multiplies the network
everywhere, not only near the boundaries: to represent a field that *decays*,
$\mathcal{N}$ must undo a factor that *grows* linearly in $t$. The condition
became exact and the function the network must learn became harder. Whether
that trade is worth it is a question about your problem, not a rule.

---

## 6 · Save

In [ ]:
os.makedirs("Ex08.2_outputs", exist_ok=True)
path = os.path.join("Ex08.2_outputs", "nb02_hard.npz")
np.savez(path, ts=ts, rel=rel, abs_err=ab, ic_err=ic_err, edge_err=edge_err,
         adam=history_hard["adam"], lbfgs=history_hard["lbfgs"])
torch.save(model_h.state_dict(), os.path.join("Ex08.2_outputs", "nb02_hard.pt"))
print("wrote", path)

## 7 · Before you move on

1. The untrained network already satisfied both conditions. Say precisely why,
   term by term in the trial solution.
2. The gap between soft and hard narrowed with time. Give the mechanism.
3. This construction needed $f_{\mathrm{IC}}$ as a **formula**. Describe what
   you would do if the initial field were a measured thermal image, and what
   new error you would be introducing.

Next: **notebook 03**, the same equation on a domain a formula cannot cover.